# 02. Extract M2D-CLAP audio + text embeddings

Reference: Niizumi et al., *M2D-CLAP: Exploring General-purpose Audio-Language Representations Beyond CLAP*, IEEE Access, 2025.

We use the journal-paper checkpoint **`m2d_clap_vit_base-80x1001p16x16p16kpBpTI-2025`** (768-dim CLAP features, 10 s @ 16 kHz, 80-mel x 1001-frame input). Make sure `python src/download_m2d_weights.py` has been run.

Outputs are saved to `results/m2d_clap/` so the comparison notebook can load both models side-by-side.


In [ ]:
import os, sys, warnings
warnings.simplefilter('ignore')
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
if str(PROJECT_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / 'src'))

DATA_CSV    = PROJECT_DIR / 'data' / 'esc50_metadata.csv'
RESULTS_DIR = PROJECT_DIR / 'results' / 'm2d_clap'
WEIGHT_FILE = (
    PROJECT_DIR / 'checkpoints' /
    'm2d_clap_vit_base-80x1001p16x16p16kpBpTI-2025' /
    'checkpoint-30.pth'
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR :', PROJECT_DIR)
print('CSV exists  :', DATA_CSV.exists())
print('weight ok   :', WEIGHT_FILE.exists())

## Load metadata + filter bad files

In [ ]:
import pandas as pd, numpy as np, soundfile as sf

df = pd.read_csv(DATA_CSV)
good = []
for p in df['audio_path']:
    try:
        data, _ = sf.read(p)
        good.append(len(data) > 0)
    except Exception:
        good.append(False)
df = df[good].reset_index(drop=True)
print('samples after filter:', len(df))
df.head()

## Load M2D-CLAP via PortableM2D

`flat_features=True` returns CLAP features (one 768-d vector per clip). On Apple Silicon we'll use the MPS backend; on a CUDA box it picks GPU automatically.

In [ ]:
import torch
from portable_m2d import PortableM2D

model = PortableM2D(weight_file=str(WEIGHT_FILE), flat_features=True)
device = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
model = model.to(device).eval()
print('device:', device)

## Audio: load, pad to 10 s @ 16 kHz, encode in batches

ESC-50 clips are 5 s at 44.1 kHz. We resample to 16 kHz and right-pad to 10 s, which matches the M2D-CLAP input contract.

In [ ]:
import librosa
from tqdm.auto import tqdm

SR, CLIP_SECONDS = 16000, 10

def load_audio_10s(path):
    y, _ = librosa.load(path, sr=SR, mono=True)
    target = SR * CLIP_SECONDS
    if y.shape[-1] >= target:
        y = y[:target]
    else:
        y = np.pad(y, (0, target - y.shape[-1]))
    return y.astype(np.float32)

audio_paths = df['audio_path'].tolist()
captions    = df['caption'].tolist()

BATCH = 16
audio_embs = []
with torch.no_grad():
    for s in tqdm(range(0, len(audio_paths), BATCH), desc='audio batches'):
        chunk = audio_paths[s:s+BATCH]
        wavs = np.stack([load_audio_10s(p) for p in chunk])
        wavs = torch.from_numpy(wavs).to(device)
        emb  = model.encode_clap_audio(wavs)
        audio_embs.append(emb.detach().cpu().float().numpy())
audio_emb = np.concatenate(audio_embs, axis=0)
print('audio_emb shape:', audio_emb.shape)

## Text: encode all per-row captions

In [ ]:
TBATCH = 32
text_embs = []
with torch.no_grad():
    for s in tqdm(range(0, len(captions), TBATCH), desc='text batches'):
        chunk = captions[s:s+TBATCH]
        emb = model.encode_clap_text(chunk, truncate=True)
        if isinstance(emb, torch.Tensor):
            emb = emb.detach().cpu().float().numpy()
        else:
            emb = np.asarray(emb, dtype=np.float32)
        text_embs.append(emb)
text_emb = np.concatenate(text_embs, axis=0)
print('text_emb shape:', text_emb.shape)
assert text_emb.shape[1] == audio_emb.shape[1], 'audio/text dims must match'

## Per-class text embeddings (50 prompts) for zero-shot

In [ ]:
class_names    = sorted(df['class_name'].unique().tolist())
class_prompts  = [f'This is a sound of {n.replace("_", " ")}.' for n in class_names]
with torch.no_grad():
    cemb = model.encode_clap_text(class_prompts, truncate=True)
    if isinstance(cemb, torch.Tensor):
        cemb = cemb.detach().cpu().float().numpy()
    else:
        cemb = np.asarray(cemb, dtype=np.float32)
print('class_text_emb shape:', cemb.shape)

## Save outputs

In [ ]:
np.save(RESULTS_DIR / 'audio_embeddings.npy', audio_emb)
np.save(RESULTS_DIR / 'text_embeddings.npy',  text_emb)
np.save(RESULTS_DIR / 'class_text_embeddings.npy', cemb)
with open(RESULTS_DIR / 'class_names.txt', 'w') as f:
    f.write('\n'.join(class_names))
df.to_csv(RESULTS_DIR / 'metadata.csv', index=False)

print('saved to', RESULTS_DIR)
for p in sorted(RESULTS_DIR.iterdir()):
    print(' ', p.name, p.stat().st_size, 'bytes')